In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/cleaned_retail_data.csv")

In [3]:
transactions = (df.groupby("Invoice")["Description"].apply(list).reset_index())
transactions.head()

,Invoice,Description
0,489434,"[15CM CHRISTMAS GLASS BALL 20 LIGHTS, PINK CHE..."
1,489435,"[CAT BOWL , DOG BOWL , CHASING BALL DESIGN, HE..."
2,489436,"[DOOR MAT BLACK FLOCK , LOVE BUILDING BLOCK WO..."
3,489437,"[CHRISTMAS CRAFT HEART DECORATIONS, CHRISTMAS ..."
4,489438,"[DINOSAURS WRITING SET , SET OF MEADOW FLOWE..."


In [4]:
df["Description"].nunique()

5283

In [5]:
(df["Description"].value_counts() < 5).sum()

609

In [6]:
transactions["Invoice"].nunique()

36969

In [7]:
avg_products_per_invoice = transactions["Description"].apply(len).mean()

In [8]:
invoice_summary = pd.DataFrame({
    "Metric": ["Average Products per Invoice"],
    "Value": [avg_products_per_invoice]
})

invoice_summary

,Metric,Value
0,Average Products per Invoice,21.083205


In [9]:
product_counts = df["Description"].value_counts()
frequent_products = product_counts[
    product_counts >= 5
].index
filtered_df = df[
    df["Description"].isin(frequent_products)
]

In [10]:
basket = (
    filtered_df
    .groupby(["Invoice", "Description"])["Quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

basket_encoded = (basket > 0).astype(int)

In [11]:
basket_encoded.shape

(36935, 4674)

In [12]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    basket_encoded,
    min_support=0.02,
    use_colnames=True,
    low_memory=True
)

c:\Python310\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [13]:
df["Description"].nunique()

5283

In [14]:
df["StockCode"].nunique()

4631

In [15]:
frequent_itemsets.head()

,support,itemsets
0,0.036849,(6 RIBBONS RUSTIC CHARM)
1,0.021687,(60 CAKE CASES VINTAGE CHRISTMAS)
2,0.047787,(60 TEATIME FAIRY CAKE CASES)
3,0.031840,(72 SWEETHEART FAIRY CAKE CASES)
4,0.028185,(ALARM CLOCK BAKELIKE GREEN)


In [16]:
top_10_itemsets = (
    frequent_itemsets
    .sort_values("support", ascending=False)
    .head(10)
)

top_10_itemsets[["itemsets", "support"]]

,itemsets,support
154,(WHITE HANGING HEART T-LIGHT HOLDER),0.132341
123,(REGENCY CAKESTAND 3 TIER),0.089806
8,(ASSORTED COLOUR BIRD ORNAMENT),0.071802
62,(JUMBO BAG RED RETROSPOT),0.070719
97,(PARTY BUNTING),0.056234
73,(LUNCH BAG BLACK SKULL.),0.054068
80,(LUNCH BAG SPACEBOY DESIGN ),0.050738
126,(REX CASH+CARRY JUMBO SHOPPER),0.050278
50,(HOME BUILDING BLOCK WORD),0.049574
146,(STRAWBERRY CERAMIC TRINKET BOX),0.049222


In [17]:
frequent_itemsets["Length"] = (
    frequent_itemsets["itemsets"]
    .apply(len)
)
top_10_pairs = (
    frequent_itemsets[
        frequent_itemsets["Length"] == 2
    ]
    .sort_values("support", ascending=False)
    .head(10)
)

top_10_pairs[["itemsets", "support"]]

,itemsets,support
171,"(WHITE HANGING HEART T-LIGHT HOLDER, RED HANGI...",0.031217
173,"(WOODEN PICTURE FRAME WHITE FINISH, WOODEN FRA...",0.026885
167,"(HEART OF WICKER LARGE, HEART OF WICKER SMALL)",0.023528
172,"(STRAWBERRY CERAMIC TRINKET BOX, SWEETHEART CE...",0.022120
168,"(HOME BUILDING BLOCK WORD, LOVE BUILDING BLOCK...",0.021281
170,"(LUNCH BAG SPACEBOY DESIGN , LUNCH BAG BLACK ...",0.020956
166,"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",0.020116
169,"(LUNCH BAG CARS BLUE, LUNCH BAG BLACK SKULL.)",0.020062


In [18]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

In [19]:
top_rules = (
    rules
    .sort_values("lift", ascending=False)
    .head(10)
)

In [20]:
top_rules["Antecedent"] = top_rules["antecedents"].apply(
    lambda x: ", ".join(list(x))
)

top_rules["Consequent"] = top_rules["consequents"].apply(
    lambda x: ", ".join(list(x))
)

In [21]:
top_10_rules = top_rules[
    [
        "Antecedent",
        "Consequent",
        "support",
        "confidence",
        "lift"
    ]
]

top_10_rules

,Antecedent,Consequent,support,confidence,lift
0,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.020116,0.796356,27.853601
1,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.020116,0.703598,27.853601
12,STRAWBERRY CERAMIC TRINKET BOX,SWEETHEART CERAMIC TRINKET BOX,0.022120,0.449395,14.174553
13,SWEETHEART CERAMIC TRINKET BOX,STRAWBERRY CERAMIC TRINKET BOX,0.022120,0.697694,14.174553
14,WOODEN PICTURE FRAME WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.026885,0.598914,12.476538
15,WOODEN FRAME ANTIQUE WHITE,WOODEN PICTURE FRAME WHITE FINISH,0.026885,0.560068,12.476538
5,LOVE BUILDING BLOCK WORD,HOME BUILDING BLOCK WORD,0.021281,0.529293,10.676917
4,HOME BUILDING BLOCK WORD,LOVE BUILDING BLOCK WORD,0.021281,0.429274,10.676917
2,HEART OF WICKER LARGE,HEART OF WICKER SMALL,0.023528,0.498566,10.351053
3,HEART OF WICKER SMALL,HEART OF WICKER LARGE,0.023528,0.488477,10.351053


In [22]:
top_rules.to_csv(
    "../models/recommendation_rules/recommendation_rules.csv",
    index=False
)